# 目标钩子

此文件包含TVM框架中目标钩子功能的单元测试，主要测试外部代码生成相关的功能。

In [1]:
import set_env

In [2]:
import sys
import numpy as np
import logging
from pathlib import Path

import tvm
import tvm.testing
from tvm import relay, IRModule
from tvm.contrib import utils
from tvm.relay.op.annotation import compiler_begin, compiler_end
from tvm import relay, runtime, testing
logging.basicConfig(level=logging.INFO)

In [3]:
def set_external_func_attr(func, compiler, ext_symbol):
    func = func.with_attr("Primitive", tvm.tir.IntImm("int32", 1))
    func = func.with_attr("Compiler", compiler)
    func = func.with_attr("global_symbol", ext_symbol)
    return func

def update_lib(lib, source_dir):
    source_dir = Path(source_dir)
    contrib_path = source_dir/"src/runtime/contrib"

    kwargs = {}
    kwargs["options"] = ["-O2", "-std=c++17", f"-I{contrib_path}"]
    tmp_path = utils.tempdir()
    lib_name = "lib.so"
    lib_path = tmp_path.relpath(lib_name)
    lib.export_library(lib_path, fcompile=False, **kwargs)
    lib = tvm.runtime.load_module(lib_path)
    return lib

def check_result(
    mod, map_inputs, out_shape, result, tol=1e-5, 
    target="llvm", device=tvm.cpu(), 
    source_dir="/media/pc/data/board/arria10/lxw/tasks/tvm"):
    with tvm.transform.PassContext(opt_level=3, disabled_pass=["AlterOpLayout"]):
        exe = relay.vm.compile(mod, target=target)
    code, lib = exe.save()
    lib = update_lib(lib, source_dir=source_dir)
    exe = runtime.vm.Executable.load_exec(code, lib)
    vm = runtime.vm.VirtualMachine(exe, device)
    out = vm.run(**map_inputs)
    tvm.testing.assert_allclose(out.numpy(), result, rtol=tol, atol=tol)

## 测试内联模式下不使用目标实例的TIR外部代码生成

此测试用例验证当不使用具体目标实例时，内联模式的TIR外部代码生成是否正常工作。

它将简单的加法算子标记为外部函数，该函数在自定义代码生成时会被替换为减法算子。

In [4]:
# 定义输入数据形状
shape = (8,)
# 生成随机测试数据
x_data = np.random.randint(255, size=shape).astype("float32")
y_data = np.random.randint(255, size=shape).astype("float32")
inputs = {"x": x_data, "y": y_data}

# 创建Relay变量
x0 = relay.var("x0", shape=shape, dtype="float32")
y0 = relay.var("y0", shape=shape, dtype="float32")
# 定义简单的加法操作
z = x0 + y0
# 创建函数
f = relay.Function([x0, y0], z)
# 设置外部函数属性，指定目标钩子和全局符号名称
f = set_external_func_attr(f, "example_target_hook", "replace_add_with_subtract")

# 创建主函数的输入变量
x = relay.var("x", shape=(8,), dtype="float32")
y = relay.var("y", shape=(8,), dtype="float32")
# 调用外部函数
call = relay.Call(f, [x, y])
# 创建IR模块
func = IRModule.from_expr(call)

# 验证结果，预期输出为x_data - y_data（因为加法被替换为减法）
check_result(func, inputs, (8,), x_data - y_data)
func.show()

## 测试带有目标实例的轮廓模式下的TIR外部代码生成

此测试用例验证当使用具体目标实例（包括自定义属性）时，轮廓模式的TIR外部代码生成是否正常工作。
    
它演示了如何将目标属性传递到自定义pass中。

In [5]:
# 定义输入数据形状
shape = (8,)
# 生成随机测试数据
x_data = np.random.randint(255, size=shape).astype("float32")
y_data = np.random.randint(255, size=shape).astype("float32")
inputs = {"x": x_data, "y": y_data}
# 使用带钩子的目标实例进行编译，以演示将目标属性传递到自定义pass中
host_target = tvm.target.Target("llvm")
generic_target = tvm.target.Target("llvm", host=host_target)
# 创建带有自定义属性的外部代码生成目标
extern_codegen_target = tvm.target.Target(
    "example_target_hook -example_attribute=42", host=host_target
)
# 从文本创建IR模块
mod = tvm.relay.fromtext(
    """
        #[version = "0.0.5"]
        def @main(%x: Tensor[(8), float32], %y: Tensor[(8), float32]) -> Tensor[(8), float32] {
            @replace_add_with_subtract(%x, %y) * 2.0f
        }

        def @replace_add_with_subtract(%x: Tensor[(8), float32], %y: Tensor[(8), float32],
                                        Inline=1,
                                        Primitive=1,
                                        Compiler="example_target_hook",
                                        global_symbol="replace_add_with_subtract") -> Tensor[(8), float32] {
            %x + %y  // 将会被自定义pass重写为实现%x - %y - 42.0f的TIR
        }
    """
)

# 验证结果，预期输出为(x_data - y_data - 42.0) * 2.0
check_result(
    mod,
    inputs,
    (8,),
    (x_data - y_data - 42.0) * 2.0,
    target=[generic_target, extern_codegen_target],
)
mod.show()

INFO:te_compiler:Using injective.cpu for multiply based on highest priority (10)


## 测试运行时模块生成

此测试用例验证带有 `'tir_to_runtime'` 属性的函数是否能正确触发 TIR 到运行时的代码生成流程。

In [6]:
# 定义输入数据形状
shape = (8,)
# 生成随机测试数据
x_data = np.random.randint(255, size=shape).astype("float32")
y_data = np.random.randint(255, size=shape).astype("float32")
inputs = {"x": x_data, "y": y_data}

# 创建Relay变量
x0 = relay.var("x0", shape=shape, dtype="float32")
y0 = relay.var("y0", shape=shape, dtype="float32")
# 定义简单的加法操作
z = x0 + y0
# 创建函数
func = relay.Function([x0, y0], z)
# 设置外部函数属性
func = set_external_func_attr(func, "example_target_hook", "replace_add_with_subtract")
# 添加触发TIRToRuntime代码生成的钩子
func = func.with_attr("tir_to_runtime", True)

# 创建主函数的输入变量
x = relay.var("x", shape=(8,), dtype="float32")
y = relay.var("y", shape=(8,), dtype="float32")
# 调用外部函数
call = relay.Call(func, [x, y])
# 创建IR模块
func = IRModule.from_expr(call)

# 验证结果，预期输出为x_data * y_data
check_result(func, inputs, (8,), x_data * y_data)
func.show()